In [28]:
import sys
from pathlib import Path

def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False

def clone_repository() -> None:
    !git clone https://github.com/jianghongab/mlfs-book.git
    %cd mlfs-book

def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml

if is_google_colab():
    clone_repository()
    install_dependencies()
    root_dir = str(Path().absolute())
    print("Google Colab environment")
else:
    root_dir = Path().absolute()
    if root_dir.parts[-1:] == ('pollen',):
        root_dir = Path(*root_dir.parts[:-1])
    if root_dir.parts[-1:] == ('notebooks',):
        root_dir = Path(*root_dir.parts[:-1])
    root_dir = str(root_dir) 
    print("Local environment")

# Add the root directory to the `PYTHONPATH` to use the `recsys` Python module from the notebook.
if root_dir not in sys.path:
    sys.path.append(root_dir)
print(f"Added the following directory to the PYTHONPATH: {root_dir}")
    
# Set the environment variables from the file <root_dir>/.env
from mlfs import config
settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

Local environment
Added the following directory to the PYTHONPATH: /Users/hongjiang/git/mlfs-book
HopsworksSettings initialized!


In [29]:
import datetime
import pandas as pd
import xgboost as xgb
import hopsworks
import json
from mlfs.airquality import util
import os
import joblib

In [30]:
project = hopsworks.login(engine="python")
fs = project.get_feature_store()

secrets = hopsworks.get_secrets_api()
location_str = secrets.get_secret("SENSOR_LOCATION_JSON").value
location = json.loads(location_str)
city = location['city']
latitude = location['latitude']
longitude = location['longitude']


2026-01-11 22:37:07,744 INFO: Closing external client and cleaning up certificates.
2026-01-11 22:37:07,746 INFO: Connection closed.
2026-01-11 22:37:07,747 INFO: Initializing external client
2026-01-11 22:37:07,748 INFO: Base URL: https://c.app.hopsworks.ai:443
2026-01-11 22:37:08,322 WARNING: UserWarning: The installed hopsworks client version 4.6.0 may not be compatible with the connected Hopsworks backend version 4.2.2. 
To ensure compatibility please install the latest bug fix release matching the minor version of your backend (4.2) by running 'pip install hopsworks==4.2.*'



2026-01-11 22:37:09,080 INFO: Python Engine initialized.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/1292436


In [31]:
# Configuration: Set number of days to forecast (can be changed to 3, 7, 14, etc.)
# You can also set this in your .env file as FORECAST_DAYS=7
FORECAST_DAYS = int(os.getenv('FORECAST_DAYS', 7))
print(f"Will forecast for the next {FORECAST_DAYS} days")

Will forecast for the next 7 days


In [32]:
today = datetime.datetime.now() - datetime.timedelta(0)
# Generate list of dates for the next N days
forecast_dates = [today + datetime.timedelta(days=i) for i in range(1, FORECAST_DAYS + 1)]
print(f"Forecast dates: {[d.strftime('%Y-%m-%d') for d in forecast_dates]}")

Forecast dates: ['2026-01-12', '2026-01-13', '2026-01-14', '2026-01-15', '2026-01-16', '2026-01-17', '2026-01-18']


In [33]:

# 1. Get model
mr = project.get_model_registry()
retrieved_model = mr.get_model(name="grass_pollen_model", version=3)

# 2. Download model
saved_model_dir = retrieved_model.download()
print(f"📦 Model downloaded to: {saved_model_dir}")

# 3. Auto-detect and load model
file_list = os.listdir(saved_model_dir)
print(f"Contains files: {file_list}")

if "pollen_model.pkl" in file_list:
    model_path = os.path.join(saved_model_dir, "pollen_model.pkl")
    model = joblib.load(model_path)
    print("✅ Successfully loaded .pkl model")
elif "model.json" in file_list:
    model_path = os.path.join(saved_model_dir, "model.json")
    model = xgb.XGBRegressor()
    model.load_model(model_path)
    print("✅ Successfully loaded .json model")

Downloading: 0.000%|          | 0/195267 elapsed<00:00 remaining<?

📦 Model downloaded to: /var/folders/vv/h58bs1s95plbyzd7qh_6vd1c0000gn/T/c2d979a5-632b-41fd-991c-712f765bc819/grass_pollen_model/3
Contains files: ['pollen_model.pkl']
✅ Successfully loaded .pkl model


In [34]:
hourly_df = util.get_hourly_weather_forecast(city, latitude, longitude)
hourly_df = hourly_df.set_index('date')



# We will only make 1 daily prediction, so we will replace the hourly forecasts with a single daily forecast
# We only want the daily weather data, so only get weather at 12:00
daily_df = hourly_df.between_time('11:59', '12:01')
daily_df = daily_df.reset_index()
daily_df['date'] = pd.to_datetime(daily_df['date']).dt.strftime('%Y-%m-%d %H:%M:%S')
daily_df = daily_df.rename(columns={'date': 'datetime_id'}) # Rename column
daily_df['city'] = city
daily_df



Coordinates 59.25°N 18.0°E
Elevation 24.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s


,datetime_id,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant,city
0,2026-01-04 12:00:00,-5.526500,0.0,19.813087,24.702408,Stockholm
1,2026-01-05 12:00:00,-7.326500,0.0,10.630672,28.300659,Stockholm
2,2026-01-06 12:00:00,-1.926500,0.1,3.415260,71.564964,Stockholm
3,2026-01-07 12:00:00,-7.976501,0.0,6.915374,231.340164,Stockholm
4,2026-01-08 12:00:00,0.173500,0.0,21.674870,168.503464,Stockholm
5,2026-01-09 12:00:00,-2.976500,0.0,14.113653,84.144089,Stockholm
6,2026-01-10 12:00:00,-5.526500,0.0,21.962950,359.060822,Stockholm
7,2026-01-11 12:00:00,-3.426500,0.0,24.014996,347.005371,Stockholm
8,2026-01-12 12:00:00,-3.926500,0.0,6.162207,353.290253,Stockholm
9,2026-01-13 12:00:00,-1.876500,0.0,9.585739,124.286934,Stockholm


In [35]:

df_recent = daily_df

# 3. Perform feature engineering locally (no longer dependent on weather_fg.read())
df_recent = df_recent.sort_values('datetime_id')

# Convert datetime_id to datetime for calculations
df_recent['datetime_id'] = pd.to_datetime(df_recent['datetime_id'])

# --- Calculate time features ---
df_recent['day_of_year'] = df_recent['datetime_id'].dt.dayofyear
df_recent['month'] = df_recent['datetime_id'].dt.month
df_recent['is_high_season'] = df_recent['day_of_year'].apply(lambda x: 1 if 140 <= x <= 250 else 0)

# --- Calculate GDD ---
T_base = 5.0
df_recent['gdd_daily'] = df_recent['temperature_2m_mean'].apply(lambda t: max(0, t - T_base))
df_recent['gdd_cumsum'] = df_recent.groupby(df_recent['datetime_id'].dt.year)['gdd_daily'].cumsum()

# --- Calculate key lag features ---
df_recent['precip_lag_1'] = df_recent['precipitation_sum'].shift(1)
df_recent['temp_lag_1'] = df_recent['temperature_2m_mean'].shift(1)
df_recent['wind_lag_1'] = df_recent['wind_speed_10m_max'].shift(1)

yesterday = today - datetime.timedelta(days=1)
pollen_df = util.get_historical_pollen(
    start_date=yesterday.strftime('%Y-%m-%d'),
    end_date=today.strftime('%Y-%m-%d')
)
# get latest pollen value 
if not pollen_df.empty:
    latest_pollen = pollen_df.sort_values('date').iloc[-1]['grass_pollen']
else:
    latest_pollen = 0  # default value 

df_recent['pollen_lag_1'] = latest_pollen

# 4. Filter prediction targets: keep only rows from today onwards
batch_data = df_recent[df_recent['datetime_id'] >= datetime.datetime.now().strftime('%Y-%m-%d')].dropna()



No pollen data available for 2026-01-10 to 2026-01-11


In [36]:
feature_cols = ['temperature_2m_mean', 'precipitation_sum', 'wind_speed_10m_max', 'wind_direction_10m_dominant',  'day_of_year', 'month', 'is_high_season', 'gdd_daily', 'gdd_cumsum', 'precip_lag_1', 'temp_lag_1', 'wind_lag_1']
batch_data[feature_cols]


batch_data['predicted_pollen'] = model.predict(batch_data[feature_cols])
batch_data

,datetime_id,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant,city,day_of_year,month,is_high_season,gdd_daily,gdd_cumsum,precip_lag_1,temp_lag_1,wind_lag_1,pollen_lag_1,predicted_pollen
7,2026-01-11 12:00:00,-3.4265,0.0,24.014996,347.005371,Stockholm,11,1,0,0,0,0.0,-5.5265,21.962950,0,0.011355
8,2026-01-12 12:00:00,-3.9265,0.0,6.162207,353.290253,Stockholm,12,1,0,0,0,0.0,-3.4265,24.014996,0,0.009772
9,2026-01-13 12:00:00,-1.8765,0.0,9.585739,124.286934,Stockholm,13,1,0,0,0,0.0,-3.9265,6.162207,0,0.007751
10,2026-01-14 12:00:00,-0.0765,0.0,23.469128,147.528824,Stockholm,14,1,0,0,0,0.0,-1.8765,9.585739,0,0.012965
11,2026-01-15 12:00:00,1.1235,0.1,13.783817,139.236481,Stockholm,15,1,0,0,0,0.0,-0.0765,23.469128,0,0.000318
12,2026-01-16 12:00:00,2.3735,0.0,14.408997,167.005386,Stockholm,16,1,0,0,0,0.1,1.1235,13.783817,0,-0.000979
13,2026-01-17 12:00:00,1.1235,0.0,14.578888,159.775055,Stockholm,17,1,0,0,0,0.0,2.3735,14.408997,0,0.005578


In [37]:
batch_data['city'] = city
batch_data['days_before_forecast_day'] = range(1, len(batch_data)+1)
batch_data = batch_data.sort_values(by=['datetime_id'])
batch_data

,datetime_id,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant,city,day_of_year,month,is_high_season,gdd_daily,gdd_cumsum,precip_lag_1,temp_lag_1,wind_lag_1,pollen_lag_1,predicted_pollen,days_before_forecast_day
7,2026-01-11 12:00:00,-3.4265,0.0,24.014996,347.005371,Stockholm,11,1,0,0,0,0.0,-5.5265,21.962950,0,0.011355,1
8,2026-01-12 12:00:00,-3.9265,0.0,6.162207,353.290253,Stockholm,12,1,0,0,0,0.0,-3.4265,24.014996,0,0.009772,2
9,2026-01-13 12:00:00,-1.8765,0.0,9.585739,124.286934,Stockholm,13,1,0,0,0,0.0,-3.9265,6.162207,0,0.007751,3
10,2026-01-14 12:00:00,-0.0765,0.0,23.469128,147.528824,Stockholm,14,1,0,0,0,0.0,-1.8765,9.585739,0,0.012965,4
11,2026-01-15 12:00:00,1.1235,0.1,13.783817,139.236481,Stockholm,15,1,0,0,0,0.0,-0.0765,23.469128,0,0.000318,5
12,2026-01-16 12:00:00,2.3735,0.0,14.408997,167.005386,Stockholm,16,1,0,0,0,0.1,1.1235,13.783817,0,-0.000979,6
13,2026-01-17 12:00:00,1.1235,0.0,14.578888,159.775055,Stockholm,17,1,0,0,0,0.0,2.3735,14.408997,0,0.005578,7


In [38]:
monitor_fg = fs.get_or_create_feature_group(
    name='grass_pollen_predictions',
    description='Grass pollen prediction monitoring',
    version=1,
    primary_key=['city','datetime_id','days_before_forecast_day']
)

In [39]:
monitor_fg.insert(batch_data, wait=True)

Uploading Dataframe: 100.00% |█| Rows 7/7 | Elapsed Time: 00:01 | Remaining Time: 00:00


Launching job: grass_pollen_predictions_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai:443/p/1292436/jobs/named/grass_pollen_predictions_1_offline_fg_materialization/executions
2026-01-11 22:37:30,371 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-01-11 22:37:33,559 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-01-11 22:39:47,566 INFO: Waiting for execution to finish. Current state: SUCCEEDING. Final status: UNDEFINED
2026-01-11 22:39:50,774 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: SUCCEEDED
2026-01-11 22:39:50,940 INFO: Waiting for log aggregation to finish.
2026-01-11 22:40:06,444 INFO: Execution finished successfully.


(Job('grass_pollen_predictions_1_offline_fg_materialization', 'SPARK'), None)